In [26]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [27]:
terra_disponivel = pd.read_excel('exerc3.xlsx',sheet_name='terra_disponivel', index_col=0).to_dict()
demanda_producao = pd.read_excel('exerc3.xlsx',sheet_name='demanda_producao', index_col=0).to_dict()
horas_trabalho = pd.read_excel('exerc3.xlsx',sheet_name='horas_trabalho', index_col=0).T.to_dict()
custo_trabalho = pd.read_excel('exerc3.xlsx',sheet_name='custo_trabalho', index_col=0).to_dict()


In [28]:
terra_disponivel

{'terra_disponivel': {'inglaterra': 70, 'frança': 110, 'espanha': 80}}

In [29]:
demanda_producao

{'demanda': {'trigo': 125, 'cevada': 60, 'aveia': 75}}

In [30]:
horas_trabalho

{'inglaterra': {'trigo': 18, 'cevada': 15, 'aveia': 12},
 'frança': {'trigo': 13, 'cevada': 12, 'aveia': 10},
 'espanha': {'trigo': 16, 'cevada': 12, 'aveia': 16}}

In [21]:
custo_trabalho

{'inglaterra': {'trigo': 3, 'cevada': 3, 'aveia': 2},
 'frança': {'trigo': 2, 'cevada': 3, 'aveia': 3},
 'espanha': {'trigo': 3, 'cevada': 3, 'aveia': 2}}

In [45]:
model = pyo.ConcreteModel()

model.paises = pyo.Set(initialize=horas_trabalho.keys())
model.alimentos = pyo.Set(initialize=demanda_producao['demanda'].keys())
model.x = pyo.Var(model.paises, model.alimentos, bounds=(0,None), domain=NonNegativeReals)


def objetivo(model):

    return sum(
        model.x[p,a] * custo_trabalho[p][a] * horas_trabalho[p][a] for p in model.paises for a in model.alimentos
    )
model.obj = pyo.Objective(rule=objetivo, sense=pyo.minimize)

def restricao_terra_total(model,p):
    return sum(
        model.x[p,a] for a in model.alimentos
    ) <= terra_disponivel['terra_disponivel'][p]
model.restricao_terra_total= pyo.Constraint(model.paises, rule=restricao_terra_total)

def demanda(model, a):
    return sum(
        model.x[p,a] for p in model.paises
    ) == demanda_producao['demanda'][a]
model.demanda = pyo.Constraint(model.alimentos, rule=demanda)

In [46]:
model.pprint()

2 Set Declarations
    alimentos : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'trigo', 'cevada', 'aveia'}
    paises : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'inglaterra', 'frança', 'espanha'}

1 Var Declarations
    x : Size=9, Index=paises*alimentos
        Key                      : Lower : Value : Upper : Fixed : Stale : Domain
            ('espanha', 'aveia') :     0 :  None :  None : False :  True : NonNegativeReals
           ('espanha', 'cevada') :     0 :  None :  None : False :  True : NonNegativeReals
            ('espanha', 'trigo') :     0 :  None :  None : False :  True : NonNegativeReals
             ('frança', 'aveia') :     0 :  None :  None : False :  True : NonNegativeReals
            ('frança', 'cevada') :     0 :  None :  None : False :  True : NonNegativeReals
             ('frança', 'trigo') :    

In [47]:
opt = SolverFactory('cplex')
resultado=opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmp19ir0eth.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmpeg33x6m6.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmpeg33x6m6.pyomo.lp
Objective sense      : Minimize
Variables            :       9
Objective nonzeros   :       9
Linear constraints   :       6  [Less: 3,  Equal: 3]
  Nonzeros           :      18
  RHS nonzeros       :       6

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 24.00000         Max   : 5

In [49]:
for p in model.paises:
    for a in model.alimentos:
        print(f"Quantiadade de {p,a} :",pyo.value(model.x[p,a]))

print(f"Custo total: ",pyo.value(model.obj))

Quantiadade de ('inglaterra', 'trigo') : 0.0
Quantiadade de ('inglaterra', 'cevada') : 0.0
Quantiadade de ('inglaterra', 'aveia') : 70.0
Quantiadade de ('frança', 'trigo') : 110.0
Quantiadade de ('frança', 'cevada') : 0.0
Quantiadade de ('frança', 'aveia') : 0.0
Quantiadade de ('espanha', 'trigo') : 15.0
Quantiadade de ('espanha', 'cevada') : 60.0
Quantiadade de ('espanha', 'aveia') : 5.0
Custo total:  7580.0
